In [ ]:
# To be able to access chatGPT in python code
# We need to call chatGPT in our code.
# To call chatGPT in our code, we need something called as the API Key
# We are going to use an alternative version of ChatGPT, which is currently free
# The name is Groq
#

# Should we publicily share/display our API Key

In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

In [ ]:
pip install langchain-groq -q # -q for silent installation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.1 MB/s eta 0:00:00


In [ ]:
# How the power of models are measured?
# Number of parameters
# 8b, 16b, 70b, 120b
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
response = llm.invoke('Write a story on Black holes')
print(response.content)

In the depths of space, there existed a mysterious and elusive phenomenon known as a black hole. It was a region in space where the gravitational pull was so strong that nothing, not even light, could escape once it got too close. The black hole was like a cosmic vacuum cleaner, sucking in everything that dared to venture near.

A team of scientists, led by the brilliant and fearless Dr. Rachel Kim, had been studying the black hole for years. They had been tracking its movements, monitoring its activity, and trying to learn more about its secrets. The team had been on a mission to explore the black hole up close, to gather data and conduct experiments that would help them better understand this enigmatic phenomenon.

As they approached the black hole, the team's spaceship, the Aurora, was pulled into its gravitational field. The ship was equipped with state-of-the-art technology, including a advanced propulsion system and a sophisticated navigation system. But despite their preparation

In [ ]:
response = llm.invoke('What is 952124 * 123991? Give me the numerical answer only.')
print(response.content)

118251917884


In [ ]:
# LLMs were created with the purpose of text generation
# Option1: Lets train a new LLM specifically for maths?
# Option2: Or lets just give our calculator to LLM

In [ ]:
# Docstring
def multiply(a: int, b: int) -> int:
    """ Multiplies two integers together. Use this tool whenever you need to perform calculation"""
    print("⭐Mulitply function called")
    return a*b

In [ ]:
multiply(952124, 123991)

⭐Mulitply function called


118054806884

In [ ]:
llm_with_tools = llm.bind_tools([multiply])

In [ ]:
response = llm_with_tools.invoke('What is 952124 * 123991? Give me the numerical answer only.')
response.content

''

In [ ]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 952124, 'b': 123991},
  'id': '0vaj39e0e',
  'type': 'tool_call'}]

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [ ]:
# 1. Define the "Notebook" (State)
# We strictly define what data exists in our graph.
# 'messages' is a list.
# 'add_messages' is a special rule: "When new info comes in, just add it to the list, don't delete old stuff."
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def assistant_node(state: State):
    # Get the conversation history
    current_messages = state["messages"]

    # Ask the LLM what to do
    response = llm_with_tools.invoke(current_messages)
    # Return the response so it gets added to the 'messages' list
    return {"messages": [response]}

In [ ]:
tools = [multiply]

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [ ]:
from typing import Literal
from langgraph.graph import END
def should_continue(state: State) -> Literal["tools", END]:
    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tools"

    else:
        return END

In [ ]:
from langgraph.graph import StateGraph, START

# 1. Create a blank graph
builder = StateGraph(State)

In [ ]:
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

In [ ]:
builder.add_edge(START, "assistant")

In [ ]:
builder.add_conditional_edges(
    "assistant",
    should_continue
)

In [ ]:
builder.add_edge("tools", "assistant")

In [ ]:
graph = builder.compile()

In [ ]:
# 1. Define the user's complex request
user_query = "Who is the current president of United States?"

# 2. Run the Graph
# We stream the steps so we can see the "thinking" process live
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")
print("--- STARTING AGENT LOOP ---")

for step in graph.stream(initial_state):
    # Print the name of the node that just finished running
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")

    # Optional: Print the actual output to see what happened inside
    # print(step)

print("--- END ---")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: Who is the current president of United States?

--- STARTING AGENT LOOP ---
📍 Node 'assistant' finished.
--- END ---

🤖 Final Answer:
I'm not able to provide real-time information or updates. However, I can suggest checking the latest news or official government websites for the most current information on the President of the United States.


In [ ]:
# Task 1
# Add a web search tool, to your AI Agent.
# Take only the web search tool code from ChatGPT/Gemini/Claude
# Rest use your exisitig Langgraph structure
# Share the google colab file for me to check!!

In [ ]:
# Task 2
# Stock API -> stock
# Weather API -> weather

In [ ]:
pip install langchain-groq langgraph langchain-community duckduckgo-search -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
response = llm.invoke('What is capital of France?')
response.content

'The capital of France is Paris.'

In [ ]:
def add(a:int, b:int) -> int:
    """Adds two integers. Use this when you need to add numbers"""
    print("⭐ add() was called")
    return a + b

def multiply(a:int, b:int) -> int:
    """Multiplies two integers. Use this when you need to multiply numbers"""
    print("⭐ multiply() was called")
    return a * b

In [ ]:
llm_with_tools = llm.bind_tools([add, multiply])

In [ ]:
response = llm_with_tools.invoke("What is 5 + 3?")
response.tool_calls

[{'name': 'add',
  'args': {'a': 5, 'b': 3},
  'id': '0yeqq9983',
  'type': 'tool_call'}]

In [ ]:
response = llm_with_tools.invoke("What is 5 * 3?")
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 5, 'b': 3},
  'id': '3874hv21j',
  'type': 'tool_call'}]

In [ ]:
response = llm_with_tools.invoke("What is 10 + 2 * 5?")
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 5},
  'id': '5cm4e4kev',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': 10, 'b': 10},
  'id': 'kvw7fcjkw',
  'type': 'tool_call'}]

In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 4.8 MB/s eta 0:00:00


In [ ]:
# Now add web search tool
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

In [ ]:
result = search.invoke("Who is the CEO of Apple? Give me one word answer only")
result

"2 days ago - In January 2011, Apple's board of directors approved a third medical leave of absence requested by Jobs. During that time, Cook was responsible for most of Apple's day-to-day operations, while Jobs made most major decisions. After Jobs resigned as CEO and became chairman of the board, Cook was named the new chief executive officer of Apple on August 24, 2011. 2 days ago - The other third of the time, where we didn't reach consensus, he just let me do it my way, never said anything more about it. In 1996, Jobs's former company, Apple, was struggling, and its survival depended on completing its next operating system. After failed negotiations to purchase Be Inc., Apple eventually came to a deal with NeXT in December for $400 million; the deal was finalized in February 1997, bringing Jobs back to the company he had cofounded. Jobs became de facto chief after then-CEO Gil Amelio was ousted in July 1997. 3 days ago - He holds a bachelor’s degree in Mechanical Engineering from 

In [ ]:
tools = [add, multiply, search]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from typing import Literal

# Every langgraph WILL begin with a state (common place for everyone to work)
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def assistant_node(state: State):
    current_messages = state["messages"]
    response = llm_with_tools.invoke(current_messages)
    return {"messages": [response]}

def should_continue(state: State) -> Literal["tools", END]:
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
tool_node = ToolNode(tools)

builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile()

In [ ]:
response = graph.invoke({"messages":[("user", "Who won the last FIFA world cup?")]})
# print(response["messages"][-1].content)
print(response["messages"])

[HumanMessage(content='Who won the last FIFA world cup?', additional_kwargs={}, response_metadata={}, id='887a67ba-3f60-434a-b74d-cfe063da291f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bkhj7xmtz', 'function': {'arguments': '{"query":"last FIFA world cup winner"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 414, 'total_tokens': 435, 'completion_time': 0.029571367, 'completion_tokens_details': None, 'prompt_time': 0.021443226, 'prompt_tokens_details': None, 'queue_time': 0.088661915, 'total_time': 0.051014593}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3478-1946-7752-b814-e84fe74b55d5-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'last FIFA world cup winner'}, 'id': 'bkhj7xmtz', 'type': 'tool_call'}], in

In [ ]:
response = graph.invoke({"messages":[("user", "What is 123 * 456?")]})
print(response["messages"])

⭐ multiply() was called
[HumanMessage(content='What is 123 * 456?', additional_kwargs={}, response_metadata={}, id='11628a19-e8e4-441e-89c4-e0a0b418ef86'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rqtfqa4rp', 'function': {'arguments': '{"a":123,"b":456}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 414, 'total_tokens': 433, 'completion_time': 0.047846774, 'completion_tokens_details': None, 'prompt_time': 0.117201451, 'prompt_tokens_details': None, 'queue_time': 0.008659261, 'total_time': 0.165048225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3478-2711-7a32-bedd-d3c419317684-0', tool_calls=[{'name': 'multiply', 'args': {'a': 123, 'b': 456}, 'id': 'rqtfqa4rp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [ ]:
response = graph.invoke({"messages":[("user", "Search the current price of gold per gram, then multiply it by 50")]})
print(response["messages"])

⭐ multiply() was called
[HumanMessage(content='Search the current price of gold per gram, then multiply it by 50', additional_kwargs={}, response_metadata={}, id='42af81c9-8b0a-418a-9cf4-7c03ffa362f0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5qdnm9zhz', 'function': {'arguments': '{"query":"current price of gold per gram"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 421, 'total_tokens': 466, 'completion_time': 0.134346368, 'completion_tokens_details': None, 'prompt_time': 0.042641475, 'prompt_tokens_details': None, 'queue_time': 0.185432361, 'total_time': 0.176987843}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3478-2892-7d82-a436-081b724d319d-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'current price

In [ ]:
# Practice Challenges — Part 1
# Challenge 1 (Easy)
# Add a subtract(a, b) tool and a divide(a, b) tool.
# Bind all four math tools together and test with: "What is (100 - 20) / 4?"

In [ ]:
def assistant_node(state: State):
    current_messages = state["messages"]
    llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)
    response = llm_with_tools.invoke(current_messages)

    return {"messages": [response]}

def should_continue(state: State) -> Literal["tools", END]:
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
def subtract(a:int, b:int) -> int:
    """Subtracts two integers. Use this when you need to subtract numbers"""
    print("⭐ subtract() was called")
    return a - b

def divide(a:int, b:int) -> int:
    """Divides two integers. Use this when you need to divide numbers"""
    print("⭐ divide() was called")
    return a // b

In [ ]:
tools = [add, multiply, subtract, divide]

In [ ]:
tool_node = ToolNode(tools)

builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile()

In [ ]:
response = graph.invoke({"messages":[("user", "First, calculate 100 - 20, then divide the result by 4.")]})
print(response["messages"][-1].content)

⭐ subtract() was called
⭐ divide() was called
The final result is 20.


In [ ]:
# Challenge 2 (Medium)
# Add a get_word_count(text: str) tool that counts words in a string.
# Then ask the agent:
# "Search for a short description of the Eiffel Tower and tell me how many words are in the result."

In [ ]:
def get_word_count(text: str) -> int:
    """Count the number of words in a text string.

    Splits the text by whitespace and returns the total word count.
    Useful for analyzing text length or verifying content requirements.

    Args:
        text: The input string to count words from.
    """
    if not text or not text.strip():
        return 0

    # Split by whitespace and count
    words = text.split()
    return len(words)

In [ ]:
tools = [get_word_count, search]

In [ ]:
tool_node = ToolNode(tools)

builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile()

In [ ]:
response = graph.invoke({"messages":[("user", "Search for a short description of the Eiffel Tower and tell me how many words are in the result.")]})
print(response["messages"][-1].content)

The search result for a short description of the Eiffel Tower contains 305 words.


In [ ]:
# Challenge 3 (Hard)
# Add a convert_currency(amount: float, rate: float) tool.
# Then ask the agent: "Search the current USD to INR exchange rate,
# then convert 200 USD to INR using that rate."
# the agent has to search first, extract the number, then call your tool.

In [ ]:
def convert_currency(amount: float, rate: float) -> float:
    """Convert an amount from one currency to another using an exchange rate.

    Multiplies the amount by the exchange rate to calculate the converted value.
    Use this AFTER you have obtained the current exchange rate from a search.

    Args:
        amount: The amount in the source currency to convert (e.g., 200 USD)
        rate: The current exchange rate (e.g., 1 USD to target currency like 83.50 INR)

    Example:
        convert_currency(200, 83.50) returns 16700.0 (200 USD = 16700 INR)
    """
    converted = amount * rate
    return round(converted, 2)  # Round to 2 decimal places for currency

In [ ]:
tools = [convert_currency, search]

In [ ]:
tool_node = ToolNode(tools)

builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile()

In [ ]:
response = graph.invoke({"messages":[("user", "Search the current USD to INR exchange rate, then convert 200 USD to INR using that rate.")]})
print(response["messages"][-1].content)

The current USD to INR exchange rate is approximately 1 USD = 82.5 INR. Converting 200 USD to INR using this rate gives us 200 * 82.5 = 16500 INR.


In [ ]:
# Topic : Memory
response = graph.invoke({"messages": [("user", "My name is Vishal.")]})
response["messages"][-1].content

"Hello Vishal, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [ ]:
# Everytime graph.invoke() starts fresh, it has no idea what you said before.
response = graph.invoke({"messages": [("user", "What is my name?")]})
response["messages"][-1].content

"I don't have any information about your name. This conversation just started, and we haven't discussed any personal details yet. If you'd like to share your name, I'd be happy to chat with you."

In [ ]:
# Memory Saver
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
tools = [add, subtract, multiply, divide, convert_currency, search]
tool_node = ToolNode(tools)

In [ ]:
builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph_with_memory = builder.compile(checkpointer=memory)

In [ ]:
config1 = {"configurable": {"thread_id": "Vishal_chat"}}

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "My name is Vishal.")]},
    config=config1
)

response["messages"][-1].content

"Hello Vishal. It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What is my name?")]},
    config=config1
)

response["messages"][-1].content

'Your name is Vishal.'

In [ ]:
config = {"configurable": {"thread_id": "conversation_3"}}

turns = [
    "I want to start my career as computer vision engineer.",
    "What is the base salary of computer vision engineer in Ahmedabad.",
    "Based on what I told you, suggest one tool I should learn next as fresher."
]

for message in turns:
    response = graph_with_memory.invoke(
        {"messages": [("user", message)]},
        config=config
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: I want to start my career as computer vision engineer.
Agent: To start your career as a computer vision engineer, focus on developing strong skills in machine learning, Python, and computer science. Build a portfolio of projects that demonstrate your ability to apply these skills to real-world challenges. Consider specializing in high-demand fields like autonomous vehicles or medical imaging to accelerate your career growth. Stay updated with the latest advancements in the field and be strategic about your career path to increase your salary and opportunities.
---
User: What is the base salary of computer vision engineer in Ahmedabad.
Agent: The average base salary of a computer vision engineer in Ahmedabad is around ₹60,000 to ₹80,000 per month, with senior roles potentially exceeding ₹8-9 lakhs per year. However, salaries can vary based on skills, experience, and company, with top-paying cities and specialized roles commanding higher salaries.
---
User: Based on what I told you

In [ ]:
# Challenge 1 (Easy)
# Start a conversation where you tell the agent your favourite programming language and
# favourite food in two separate messages.
# Then in a third message, ask it to repeat both back to you.

In [ ]:
config = {"configurable": {"thread_id": "conv_4"}}

turns = [
    "My favourite programming language is javascript.",
    "My favourite food is paneer roll.",
    "Tell me about my favourite programming language and favourite food."
]

for message in turns:
    response = graph_with_memory.invoke(
        {"messages": [("user", message)]},
        config=config
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: My favourite programming language is javascript.
Agent: I'm happy to hear that your favorite programming language is JavaScript. JavaScript is a popular and versatile language used for both front-end and back-end development. It's widely used for creating interactive web pages, web applications, and mobile applications. If you have any specific questions about JavaScript or need help with a project, feel free to ask.
---
User: My favourite food is paneer roll.
Agent: That sounds delicious. Paneer roll is a popular Indian dish made with paneer (Indian cheese) wrapped in a roll, often served with various spices and chutneys. If you're looking for a recipe or want to know more about the dish, I can try to help.
---
User: Tell me about my favourite programming language and favourite food.
Agent: Your favorite programming language is JavaScript, which is a popular and versatile language used for both front-end and back-end development. It's widely used for creating interactive web pag

In [ ]:
# Challenge 2 (Medium)
# Simulate two different students having separate conversations with the agent.
# Student A says their name is Priya and she studies biology.
# Student B says his name is Ravi and he studies physics.
# Then ask both "What do I study?" and verify the agent keeps them separate.

In [ ]:
config_priya = {"configurable": {"thread_id": "priya_chat"}}

message = "I am priya and i study biology."

response = graph_with_memory.invoke(
    {"messages": [("user", message)]},
    config=config_priya
)
print(f"User: {message}")
print(f"Agent: {response['messages'][-1].content}")
print("---")

User: I am priya and i study biology.
Agent: Hello Priya, that's great that you study biology. Is there something specific you'd like to know or discuss related to biology that I can help you with?
---


In [ ]:
config_ravi = {"configurable": {"thread_id": "ravi_chat"}}

message = "I am ravi and i study physics."

response = graph_with_memory.invoke(
    {"messages": [("user", message)]},
    config=config_ravi
)
print(f"User: {message}")
print(f"Agent: {response['messages'][-1].content}")
print("---")

User: I am ravi and i study physics.
Agent: Hello Ravi, it's nice to meet you. Physics is a fascinating field of study. What areas of physics interest you the most?
---


In [ ]:
message = "What do I study?"

response = graph_with_memory.invoke(
    {"messages": [("user", message)]},
    config=config_priya
)
print(f"User: {message}")
print(f"Agent: {response['messages'][-1].content}")
print("---")

User: What do I study?
Agent: You study biology.
---


In [ ]:
message = "What do I study?"

response = graph_with_memory.invoke(
    {"messages": [("user", message)]},
    config=config_ravi
)
print(f"User: {message}")
print(f"Agent: {response['messages'][-1].content}")
print("---")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01krt7c10dexvsnw5ed3bteatb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99686, Requested 397. Please try again in 1m11.712s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Challenge 3 (Hard)
# Build a simple number tracking conversation.
# Tell the agent three numbers across three separate messages.
# Then ask it to add all three numbers together. The agent must use the add tool and its memory to answer.

In [ ]:
config = {"configurable": {"thread_id": "add_numbers"}}

turns = [
    "Remember this number '5'",
    "Remember this number '2'",
    "Remember this number '3'"
    "Add all 3 previous numbers from memory"
]

for message in turns:
    response = graph_with_memory.invoke(
        {"messages": [("user", message)]},
        config=config
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: Remember this number '5'
Agent: I'm not able to remember numbers or store information. If you'd like to perform an operation on the number 5, I can help with that. Please let me know what you'd like to do.
---


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01krt7c10dexvsnw5ed3bteatb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 785. Please try again in 11m18.239999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Small Example First - Embeddings

In [ ]:
pip install -U langchain-huggingface -q

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings # use to generate vector embeddings

In [ ]:
embedder = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2') # Model to generate vector embeddings

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
result = embedder.embed_query('What is machine learning?')
result

[-0.019954556599259377,
 0.009878002107143402,
 0.010249638929963112,
 0.02955370768904686,
 0.027186447754502296,
 -0.0192965567111969,
 -0.02412959188222885,
 -0.037735726684331894,
 -0.041054174304008484,
 -0.001474980148486793,
 -0.07607672363519669,
 0.036872074007987976,
 0.05372385308146477,
 -0.07053526490926743,
 -0.08609345555305481,
 0.02100684493780136,
 -0.04226336628198624,
 0.02844890207052231,
 -0.038375373929739,
 -0.06591919809579849,
 0.013949654996395111,
 0.01227535679936409,
 -0.06821976602077484,
 0.02862507477402687,
 0.014761180616915226,
 0.03223143890500069,
 0.008694231510162354,
 0.013062074780464172,
 -0.015726709738373756,
 0.018896568566560745,
 0.020023854449391365,
 -0.03929707035422325,
 0.01564716175198555,
 0.03339807316660881,
 -0.07339105755090714,
 0.05935782566666603,
 -0.01238581445068121,
 0.02458464726805687,
 0.020107125863432884,
 -0.020254528149962425,
 -0.050247713923454285,
 -0.09044118970632553,
 -0.015374385751783848,
 -0.0036124335601

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

e1 = embedder.embed_query('I love cricket')
e2 = embedder.embed_query('Cricket is my favourite sport')
e3 = embedder.embed_query('The stock market crashed today')

print("Similar pair", cosine_similarity([e1], [e2]))
print("Different pair", cosine_similarity([e1], [e3]))

Similar pair [[0.86108381]]
Different pair [[0.01344572]]


### Build a RAG Pipeline on a Small Document

In [ ]:
pip install -U langchain_text_splitters -q

### Build a RAG Pipeline on a Small Document

In [ ]:
pip install -U langchain_text_splitters langchain_community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # For breaking into chunks
from langchain_community.vectorstores import Chroma # Storing the vector embeddings into Vector Database
from langchain_core.documents import Document # Storing our simple (word) document

/tmp/ipykernel_17368/2614211405.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma # Storing the vector embeddings into Vector Database


In [ ]:
# Step 1: Our document (could be pdf, csv, plain text)
my_text = """
Artificial Intelligence is a branch of computer science focused on building
machines that can perform tasks that typically require human intelligence.

Machine Learning is a subset of AI where systems learn from data instead of
being explicitly programmed. The more data you give it, the better it gets.

Deep Learning is a subset of Machine Learning that uses neural networks with
many layers. It is especially good at image and speech recognition.

Natural Language Processing (NLP) is the branch of AI that deals with text
and language. ChatGPT is an example of an NLP model.

Reinforcement Learning is a type of Machine Learning where an agent learns
by taking actions and receiving rewards or punishments. It is used in
robotics and games.
"""

In [ ]:
# Step 2 : Splitting into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200)
chunks = splitter.split_text(my_text)

In [ ]:
for chunk in chunks:
    print(chunk)
    print('----------------------------------')

Artificial Intelligence is a branch of computer science focused on building
machines that can perform tasks that typically require human intelligence.
----------------------------------
Machine Learning is a subset of AI where systems learn from data instead of
being explicitly programmed. The more data you give it, the better it gets.
----------------------------------
Deep Learning is a subset of Machine Learning that uses neural networks with
many layers. It is especially good at image and speech recognition.
----------------------------------
Natural Language Processing (NLP) is the branch of AI that deals with text
and language. ChatGPT is an example of an NLP model.
----------------------------------
Reinforcement Learning is a type of Machine Learning where an agent learns
by taking actions and receiving rewards or punishments. It is used in
robotics and games.
----------------------------------


In [ ]:
pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
# Step 3: Store the chunks
docs = [Document(page_content=chunk) for chunk in chunks]
vectorStore = Chroma.from_documents(docs, embedding=embedder)

In [ ]:
retriever = vectorStore.as_retriever(search_kwargs={'k':2}) # Retrieve top 2 most matching chunks

question = 'What is deep learning?'
relevant_chunks = retriever.invoke(question)

for chunk in relevant_chunks:
    print(chunk)


page_content='Deep Learning is a subset of Machine Learning that uses neural networks with
many layers. It is especially good at image and speech recognition.'
page_content='Machine Learning is a subset of AI where systems learn from data instead of
being explicitly programmed. The more data you give it, the better it gets.'


In [ ]:
pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
# Step 5: Pass retrieved chunks to the LLM and get the answer

def answer_from_docs(question:str) -> str:
    chunks = retriever.invoke(question)

    prompt = f"""Answer the question using only the context below.

    Context:
    {chunks}

    Question:
    {question}
    """

    response = llm.invoke(prompt)
    return response.content


In [ ]:
answer_from_docs("What is the difference between AI and Machine Learning?")

'Artificial Intelligence (AI) is a branch of computer science focused on building machines that can perform tasks that typically require human intelligence. Machine Learning, on the other hand, is a subset of AI where systems learn from data instead of being explicitly programmed. This means that AI is a broader field, while Machine Learning is a specific approach within AI that enables systems to improve with more data.'

In [ ]:
llm.invoke('What is capital of France?')

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 41, 'total_tokens': 49, 'completion_time': 0.010671365, 'completion_tokens_details': None, 'prompt_time': 0.004745244, 'prompt_tokens_details': None, 'queue_time': 0.063010085, 'total_time': 0.015416609}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5378-a58c-7ce0-84ce-8607b79740b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 8, 'total_tokens': 49})

In [ ]:
answer_from_docs("What is the capital of France?")

'There is no information about the capital of France in the provided context. The context only discusses Artificial Intelligence and Machine Learning.'

In [ ]:
# Challenge 1 (Easy)
# Add 2–3 sentences about yourself (your name, what you study, your city) as the document text.
# Build the RAG pipeline on it.
# Then ask questions like "Where does this person live?"
# and confirm it answers from your text.

In [ ]:
my_details = """
My name is Vishal Mondal.

I am studying Data science with Gen AI.

I live in Ahmedabad, Gujarat, India.
"""

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
chunks = splitter.split_text(my_details)

In [ ]:
for chunk in chunks:
    print(chunk)
    print('-'*50)

My name is Vishal Mondal.
--------------------------------------------------
I am studying Data science with Gen AI.
--------------------------------------------------
I live in Ahmedabad, Gujarat, India.
--------------------------------------------------


In [ ]:
docs = [Document(page_content=chunk) for chunk in chunks]
vectorStore = Chroma.from_documents(docs, embedding=embedder)

In [ ]:
retriever = vectorStore.as_retriever(search_kwargs={'k':1})

In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
def answer_from_docs(question:str) -> str:
  """
  Use this function when user wants you to answer from context.
  """
  chunks = retriever.invoke(question)

  prompt = f"""Try answering the question from the context below. If answer not available in context give answer from your information or use other tool.

  Context:
  {chunks}

  Question:
  {question}
  """

  response = llm.invoke(prompt)
  return response.content

In [ ]:
answer_from_docs("What is my name?")

'Your name is Vishal Mondal.'

In [ ]:
answer_from_docs("What am i studying?")

'Data science with Gen AI.'

In [ ]:
answer_from_docs("Where do i live?")

'You live in Ahmedabad, Gujarat, India.'

In [ ]:
answer_from_docs("What is my favourite food?")

'There is no information about your favourite food in the given context. The context only mentions your name as Vishal Mondal.'

In [ ]:
# Challenge 2 (Super Tough):
# Create the RAG as a tool
# and add it to your langgraph
# You need to ask it 3 questions
# First question should be from doc
# Second question should be a follow up question from doc
# Third question should not be in doc, it should need web search tool

In [ ]:
my_text = """
Python is a high-level, interpreted programming language known for its simple syntax.
It was created by Guido van Rossum and first released in 1991.
Python is widely used in data science, web development, and automation.
The latest major version is Python 3, which introduced many improvements over Python 2.
Popular Python libraries include NumPy, Pandas, and Scikit-learn for data science.
"""

# "Who created Python?",
# "What did that person release and in what year?",  # needs memory of "Guido"
# "What is the current weather in Bangalore?"         # not in doc, needs web search

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
chunks = splitter.split_text(my_text)

In [ ]:
for chunk in chunks:
    print(chunk)
    print('-'*50)

Python is a high-level, interpreted programming language known for its simple syntax.
--------------------------------------------------
It was created by Guido van Rossum and first released in 1991.
--------------------------------------------------
Python is widely used in data science, web development, and automation.
--------------------------------------------------
The latest major version is Python 3, which introduced many improvements over Python 2.
--------------------------------------------------
Popular Python libraries include NumPy, Pandas, and Scikit-learn for data science.
--------------------------------------------------


In [ ]:
docs = [Document(page_content=chunk) for chunk in chunks]
vectorStore = Chroma.from_documents(docs, embedding=embedder)

In [ ]:
retriever = vectorStore.as_retriever(search_kwargs={'k':1})

In [ ]:
pip install -U ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 82.7 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

In [ ]:
tools = [answer_from_docs, search]

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from typing import Literal

# Every langgraph WILL begin with a state (common place for everyone to work)
class State(TypedDict):
    messages: Annotated[list, add_messages]

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def assistant_node(state: State):
    current_messages = state["messages"]
    response = llm_with_tools.invoke(current_messages)
    return {"messages": [response]}

def should_continue(state: State) -> Literal["tools", END]:
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_core.tools import StructuredTool

# Explicitly create a StructuredTool for answer_from_docs
answer_from_docs_tool = StructuredTool.from_function(
    func=answer_from_docs,
    name="answer_from_docs",
    description="Use this function when user wants you to answer from context."
)

# Update the tools list to include the StructuredTool instance
tools = [answer_from_docs_tool, search]
tool_node = ToolNode(tools)

In [ ]:
builder = StateGraph(State) # Blank Graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph_with_memory = builder.compile(checkpointer=memory)

In [ ]:
my_config = {"configurable": {"thread_id": "Vishal_chat"}}

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "Who created Python?")]},
    config = my_config
)

response["messages"][-1].content

'Guido van Rossum created the Python programming language.'

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "Give answer from context, Who created Python?")]},
    config = my_config
)

response["messages"][-1].content

'Guido van Rossum created the Python programming language.'

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "Give answer from context, What did that person release and in what year?")]},
    config = my_config
)

response["messages"][-1].content

'Guido van Rossum released Python in 1991.'

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What is the current weather in Bangalore?")]},
    config = my_config
)

response["messages"][-1].content

'The current weather in Bangalore is partly cloudy with a temperature of 26.1°C, feels like 27.5°C, humidity at 74%, wind blowing at 16.2 km/h, and a UV Index of 0.'

In [ ]:
# "Who created Python?",
# "What did that person release and in what year?",  # needs memory of "Guido"
# "What is the current weather in Bangalore?"         # not in doc, needs web search

## Multi Agent Systems

In [ ]:
pip install langchain-groq langgraph langchain-community duckduckgo-search chromadb langchain-huggingface pandas matplotlib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203

### Small Example - Two Specialist Agents

In [ ]:
pip install -U ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.8 MB/s eta 0:00:00


In [ ]:
# Agent 1: Researcher - Only has web search
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()

/tmp/ipykernel_1471/4197288360.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
researcher_llm = llm.bind_tools([search])

In [ ]:
#Agent 2: Calculator - only math tools
def add(a: int, b: int) -> int:
    """Adds two numbers"""
    return a+b

def multiply(a: int, b: int) -> int:
    """Multiplies two numbers"""
    return a*b

calculator_llm = llm.bind_tools([add, multiply])

In [ ]:
# Small Test - each agent only knows its own job
response = researcher_llm.invoke("Who is the CEO of Google?")
response.tool_calls

[{'name': 'duckduckgo_search',
  'args': {'query': 'CEO of Google'},
  'id': 'cwfqq9gx7',
  'type': 'tool_call'}]

In [ ]:
response = calculator_llm.invoke('What is 12 * 15?')
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 12, 'b': 15},
  'id': 's392ev001',
  'type': 'tool_call'}]

### Build the Supervisor Graph

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages] # Common message list where everybody speaks and works
    next: str # Supervisor will use to write who goes next


In [ ]:
# Supervisor - no LLM, no tools, just decides who should work
def supervisor_node(state: State):
    messages = state["messages"]
    last = messages[-1].content.lower()

    # For now simple logic, you will integrate LLM, for now I am using if/else condition
    if any(word in last for word in ['search', 'who', 'what is', 'latest', 'news', 'current']):
        return {"next": "researcher"}

    elif any(word in last for word in ['add', 'multiply', 'calculate', '+', '*', 'sum']):
        return {"next": "calculator"}
    else:
        return {"next": "end"}

In [ ]:
# Researcher node
def researcher_node(state: State):
    response = researcher_llm.invoke(state['messages'])
    return {"messages": [response]}

# Calculator Node
def calculator_node(state: State):
    response = calculator_llm.invoke(state['messages'])
    return {"messages": [response]}

In [ ]:
# Route after supervisor decision
def route_after_supervisor(state: State) -> Literal["researcher", "calculator", END]:
    if state['next'] != 'end':
        return state['next']
    else:
        return END

In [ ]:
# Tool Node for each specialist
researcher_tool_node = ToolNode([search])
calculator_tool_node = ToolNode([add, multiply])

In [ ]:
# Handle tool calls inside each specialist
def researcher_should_continue(state: State) -> Literal['researcher_tools', END]:
    if state["messages"][-1].tool_calls:
        return "researcher_tools"
    else:
        return END

def calculator_should_continue(state: State) -> Literal['calculator_tools', END]:
    if state["messages"][-1].tool_calls:
        return "calculator_tools"
    else:
        return END

In [ ]:
# Blank Graph
builder = StateGraph(State)

builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('calculator', calculator_node)
builder.add_node('researcher_tools', researcher_tool_node)
builder.add_node('calculator_tools', calculator_tool_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_after_supervisor)

builder.add_conditional_edges('researcher', researcher_should_continue)
builder.add_edge('researcher_tools', 'researcher')

builder.add_conditional_edges('calculator', calculator_should_continue)
builder.add_edge('calculator_tools', 'calculator')

multi_agent = builder.compile()

In [ ]:
# Test - should go to researcher
response = multi_agent.invoke({"messages": [("user", "Who is the current CEO of Google?")]})
response['messages'][-1].content

'The current CEO of Google is Sundar Pichai.'

In [ ]:
response = multi_agent.invoke({"messages": [("user", "Calculate 25 * 48")]})
response['messages'][-1].content

'The result of 25 * 48 is 1200.'

In [ ]:
# Challenge 1
# Make the supervisor smarter — instead of keyword matching,
# pass the user message to the LLM and
# ask it to return just one word: "researcher", "calculator", or "end".
# Use that word for routing.

# Step 1: Take out last message from state
# Step 2: prompt = You are a supervisor, read the user message. Reply with only one word nothing else:
#  a. __________- -> return 'researcher'
#  b. __________- -> return 'calculator'
#  c. ________  -> return 'end'

#  user_message : {user_message}

#  decision = response.content

#  if decision in [researcher, calculator]:
#     return {next: decision}

# else:
#     return "end"

In [ ]:
# Supervisor - no LLM, no tools, just decides who should work
def supervisor_node(state: State):
    messages = state["messages"]
    last = messages[-1].content.lower()

    prompt = f"""You are a supervisor, read the user message. Reply with only one word nothing else:
    a. If user wants to know something that requires web search- -> return 'researcher'
    b. If user wants to do arithmetic calculation- -> return 'calculator'
    c. If none of the above tools are required- -> return 'end'

    user_message : {last}
    """

    response = llm.invoke(prompt)
    decision = response.content
    print(decision)

    if decision in ["researcher", "calculator"]:
      return {"next": decision}
    else:
      return {"next": "end"}

In [ ]:
# Researcher node
def researcher_node(state: State):
    response = researcher_llm.invoke(state['messages'])
    return {"messages": [response]}

# Calculator Node
def calculator_node(state: State):
    response = calculator_llm.invoke(state['messages'])
    return {"messages": [response]}

In [ ]:
# Route after supervisor decision
def route_after_supervisor(state: State) -> Literal["researcher", "calculator", END]:
    if state['next'] != 'end':
        return state['next']
    else:
        return END

In [ ]:
# Tool Node for each specialist
researcher_tool_node = ToolNode([search])
calculator_tool_node = ToolNode([add, multiply])

In [ ]:
# Handle tool calls inside each specialist
def researcher_should_continue(state: State) -> Literal['researcher_tools', END]:
    if state["messages"][-1].tool_calls:
        return "researcher_tools"
    else:
        return END

def calculator_should_continue(state: State) -> Literal['calculator_tools', END]:
    if state["messages"][-1].tool_calls:
        return "calculator_tools"
    else:
        return END

In [ ]:
# Blank Graph
builder = StateGraph(State)

builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('calculator', calculator_node)
builder.add_node('researcher_tools', researcher_tool_node)
builder.add_node('calculator_tools', calculator_tool_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_after_supervisor)

builder.add_conditional_edges('researcher', researcher_should_continue)
builder.add_edge('researcher_tools', 'researcher')

builder.add_conditional_edges('calculator', calculator_should_continue)
builder.add_edge('calculator_tools', 'calculator')

multi_agent = builder.compile()

In [ ]:
response = multi_agent.invoke({"messages": [("user", "Who won the IPL 2025?")]})
response['messages'][-1].content

researcher


'The winner of the IPL 2025 is Royal Challengers Bengaluru (RCB).'

In [ ]:
response = multi_agent.invoke({"messages": [("user", "What is 2 * 3?")]})
response['messages'][-1].content

calculator


'The result of 2 * 3 is 6.'

In [ ]:
# Challenge 2 (Easy)
# Add a third specialist: a greeter_node that uses plain LLM (no tools) and just responds warmly.
# Update the supervisor to route greetings

In [ ]:
# Supervisor - LLM, no tools, just decides who should work
def supervisor_node(state: State):
    messages = state["messages"]
    last = messages[-1].content.lower()

    prompt = f"""You are a supervisor, read the user message. Reply with only one word nothing else:
    a. If user wants to know something that requires web search- -> return 'researcher'
    b. If user wants to do arithmetic calculation- -> return 'calculator'
    c. If user sends a greeting message or tells about himself/herself- -> return 'greeting'
    d. If user asks about previous questions or conversation history- -> return 'history'
    e. If none of the above tools are required- -> return 'end'

    user_message : {last}
    """

    response = llm.invoke(prompt)
    decision = response.content
    print(decision)

    if decision in ["researcher", "calculator", "greeting", "history"]:
      return {"next": decision}
    else:
      return {"next": "end"}

In [ ]:
# Researcher node
def researcher_node(state: State):
    response = researcher_llm.invoke(state['messages'])
    return {"messages": [response]}

# Calculator Node
def calculator_node(state: State):
    response = calculator_llm.invoke(state['messages'])
    return {"messages": [response]}

# Greeting node
def greeting_node(state: State):
    last = state["messages"][-1].content.lower()

    prompt = f"""Greet user warmly
    user_message: {last}
    """

    response = llm.invoke(prompt)
    return {"messages": [response]}

# History node
def history_node(state: State):
    messages = state["messages"]
    # Exclude the current question itself
    previous_messages = messages[:-1]

    if not previous_messages:
        response = "You haven't asked any questions yet."
    else:
        questions = []
        for msg in previous_messages:
            if msg.type == "human":  # user messages
                questions.append(f"- {msg.content}")

        if questions:
            response = "Here are the questions you asked so far:\n" + "\n".join(questions)
        else:
            response = "I don't see any previous questions."

    return {"messages": [response]}

In [ ]:
# Route after supervisor decision
def route_after_supervisor(state: State) -> Literal["researcher", "calculator", "greeting", "history", END]:
    if state['next'] != 'end':
        return state['next']
    else:
        return END

In [ ]:
# Tool Node for each specialist
researcher_tool_node = ToolNode([search])
calculator_tool_node = ToolNode([add, multiply])

In [ ]:
# Handle tool calls inside each specialist
def researcher_should_continue(state: State) -> Literal['researcher_tools', END]:
    if state["messages"][-1].tool_calls:
        return "researcher_tools"
    else:
        return END

def calculator_should_continue(state: State) -> Literal['calculator_tools', END]:
    if state["messages"][-1].tool_calls:
        return "calculator_tools"
    else:
        return END

In [ ]:
# Blank Graph
builder = StateGraph(State)

builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('calculator', calculator_node)
builder.add_node('greeting', greeting_node)
builder.add_node('researcher_tools', researcher_tool_node)
builder.add_node('calculator_tools', calculator_tool_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_after_supervisor)

builder.add_conditional_edges('researcher', researcher_should_continue)
builder.add_edge('researcher_tools', 'researcher')

builder.add_conditional_edges('calculator', calculator_should_continue)
builder.add_edge('calculator_tools', 'calculator')

builder.add_edge('greeting', END)

multi_agent = builder.compile()

In [ ]:
response = multi_agent.invoke({"messages": [("user", "Hello, My name is Vishal.")]})
response['messages'][-1].content

greeting


"Hello Vishal, it's an absolute pleasure to meet you. I hope you're having a fantastic day so far. Is there something I can help you with or would you like to chat for a bit?"

In [ ]:
# Challenge 3 (Hard)
# Add memory to the multi-agent graph.
# Start a conversation where you first ask a math question,
# then ask a search question, then ask the agent to summarise what it helped you with across both turns.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
# Blank Graph
builder = StateGraph(State)

builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('calculator', calculator_node)
builder.add_node('greeting', greeting_node)
builder.add_node('history', history_node)
builder.add_node('researcher_tools', researcher_tool_node)
builder.add_node('calculator_tools', calculator_tool_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_after_supervisor)

builder.add_conditional_edges('researcher', researcher_should_continue)
builder.add_edge('researcher_tools', 'researcher')

builder.add_conditional_edges('calculator', calculator_should_continue)
builder.add_edge('calculator_tools', 'calculator')

builder.add_edge('greeting', END)
builder.add_edge('history', END)

multi_agent_with_memory = builder.compile(checkpointer=memory)

In [ ]:
my_config = {"configurable": {"thread_id": "new_chat2"}}

In [ ]:
turns = [
    "What is 2 * 3?",
    "What is chicken pox?",
    "What questions did i asked you till now?"
]

for message in turns:
    response = multi_agent_with_memory.invoke(
        {"messages": [("user", message)]},
        config=my_config
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

calculator
User: What is 2 * 3?
Agent: The result of 2 * 3 is 6.
---
researcher
User: What is chicken pox?
Agent: Chickenpox, also known as varicella, is a highly contagious disease caused by the varicella-zoster virus. It mainly affects kids, but adults can also get it. The disease brings on an itchy rash with small, fluid-filled blisters and spreads very easily to people who haven't had the disease or haven't gotten the chickenpox vaccine. The incubation period is 10–21 days, after which the characteristic rash appears. It may be spread from one to two days before the rash appears until all lesions have crusted over. Symptoms of chickenpox usually happen in the following order: low-grade fever, feeling tired, headache, a stomachache that makes you not want to eat, and a skin rash that’s very itchy and looks like many small blisters. Complications from chickenpox are unlikely but possible and may include bacterial infections, encephalitis or Reye’s syndrome, pneumonia, dehydration, is

In [ ]:
turns = [
    "What is 2 * 3?",
    "What is chicken pox?",
    "What questions did i asked you till now?"
]

for message in turns:
    response = multi_agent_with_memory.invoke(
        {"messages": [("user", message)]},
        config=my_config
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

calculator
User: What is 2 * 3?
Agent: The result of 2 * 3 is 6.
---
researcher
User: What is chicken pox?
Agent: Chickenpox is a highly contagious disease caused by the varicella-zoster virus (VZV). It mainly affects kids, but adults can also get it. The disease brings on an itchy rash with small, fluid-filled blisters and spreads very easily to people who haven't had the disease or haven't gotten the chickenpox vaccine. The incubation period is 10–21 days, after which the characteristic rash appears. It may be spread from one to two days before the rash appears until all lesions have crusted over. Symptoms of chickenpox usually happen in the following order: low-grade fever, feeling tired, headache, a stomachache that makes you not want to eat, and a skin rash that’s very itchy and looks like many small blisters. Complications from chickenpox are unlikely but possible and may include bacterial infections, encephalitis or Reye’s syndrome, pneumonia, dehydration, issues with how your b

In [ ]:
response = multi_agent_with_memory.invoke(
        {"messages": [("user", "What is 2 * 4?")]},
        config=my_config
    )

print(response['messages'][-1].content)

calculator
The result of 2 * 4 is 8.
